# 🏆 FIFA World Cup 2026 — Modelling
### `03_modelling.ipynb` · WC2026 Analytics & Prediction Pipeline

---

**Input :** `wc2026_features.csv` + `selected_features.json` (from `02_feature_engineering.ipynb`)  
**Fallback :** Raw `wc2026_ml_dataset.csv` with inline feature engineering if saved files are unavailable  
**Output :** `wc2026_model_results.csv`, `wc2026_final_predictions.csv`, trained model artefacts

**Notebook flow:**
1. Setup & Data Loading  
2. Train / Test Split & Cross-Validation Strategy  
3. Baseline Model  
4. Model Zoo — 6 Classifiers  
5. Hyperparameter Tuning (XGBoost + Random Forest)  
6. Model Comparison & Selection  
7. Final Model — Full Evaluation  
8. SHAP Explainability  
9. Confederation-Level Analysis  
10. World Cup 2026 Predictions  
11. Results Export  

## 1 · Setup & Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import warnings
import os
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── Scikit-learn ──────────────────────────────────────────────────────────
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, cross_validate,
    GridSearchCV, RandomizedSearchCV, train_test_split
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, roc_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── XGBoost ───────────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False
    print("⚠️  XGBoost not found — pip install xgboost. GBM will be used as fallback.")

# ── SHAP ──────────────────────────────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP available")
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️  SHAP not found — pip install shap. Section 8 will be skipped.")

# ── Plot theme ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'         : 130,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.titlesize'     : 13,
    'axes.labelsize'     : 11,
    'xtick.labelsize'    : 9,
    'ytick.labelsize'    : 9,
    'legend.fontsize'    : 9,
    'font.family'        : 'sans-serif',
})

PALETTE_QUAL  = ['#2196F3', '#FF5722']   # Not qualified / Qualified
PALETTE_CONF  = sns.color_palette('tab10', 6)
CONF_ORDER    = ['UEFA', 'CAF', 'AFC', 'CONCACAF', 'CONMEBOL', 'OFC']
TARGET        = 'qualified_for_wc2026'
RANDOM_STATE  = 42

print("\n✅ All imports complete.")

## 2 · Data Loading

We prefer the engineered dataset from `02_feature_engineering.ipynb`.  
If it doesn't exist, we rebuild features inline from the raw CSV.

In [ ]:
def build_features_inline(df_raw):
    """Rebuild engineered features if wc2026_features.csv is unavailable."""
    df = df_raw.copy()

    # Group A — Relative ranking
    conf_rank_mean = df.groupby('confederation')['fifa_rank'].transform('mean')
    conf_rank_std  = df.groupby('confederation')['fifa_rank'].transform('std')
    df['rank_vs_conf']  = -(df['fifa_rank'] - conf_rank_mean) / (conf_rank_std + 1e-9)
    df['elo_vs_conf']   = df['elo_rating'] - df.groupby('confederation')['elo_rating'].transform('mean')
    df['rank_ratio']    = df['fifa_points'] / df['fifa_points'].mean()

    # Group B — Form & Momentum
    df['form_trajectory']  = df['win_rate_last5'] - df['win_rate_last20']
    df['form_consistency'] = 1 - df[['win_rate_last5','win_rate_last10','win_rate_last20']].std(axis=1)
    peak_mask              = (df['avg_player_age'] >= 25) & (df['avg_player_age'] <= 28)
    df['peak_age_form']    = df['win_rate_last10'] * np.where(peak_mask, 1.15, 1.0)

    # Group C — Attack/Defence
    df['goal_ratio']             = df['avg_goals_scored_last10'] / (df['avg_goals_conceded_last10'] + 1e-9)
    df['defensive_reliability']  = df['clean_sheet_rate'] * df['squad_depth_score']
    df['clutch_factor']          = 1 - (df['draw_rate_last10'] + (1 - df['win_rate_last5']))

    # Group D — Squad Quality
    df['quality_adjusted_wins'] = df['win_rate_last10'] * (df['avg_opponent_elo_last10'] / 1600)
    from sklearn.preprocessing import MinMaxScaler
    tsi_feats = ['elo_rating','squad_value_total_m','win_rate_last10','xg_difference','squad_depth_score']
    df['team_strength_index']   = MinMaxScaler().fit_transform(df[tsi_feats]).mean(axis=1)

    # Group E — History & Coach
    df['historical_pedigree'] = (
        df['world_cup_appearances'] * 0.5
        + df['world_cup_titles']    * 2.0
        + df['continental_titles']  * 1.0
    )
    df['experience_score']  = df['world_cup_appearances'] * (1 + df['world_cup_titles'])
    df['coach_efficiency']  = df['coach_success_rate'] * np.log1p(df['coach_tenure_days'])
    df['conf_strength']     = df['confederation'].map(
        df.groupby('confederation')['elo_rating'].mean().to_dict()
    )
    return df


# ── Load data ──────────────────────────────────────────────────────────────
FEATURES_PATH     = 'wc2026_features.csv'
FEAT_LIST_PATH    = 'selected_features.json'
RAW_PATH          = 'wc2026_ml_dataset.csv'

if os.path.exists(FEATURES_PATH) and os.path.exists(FEAT_LIST_PATH):
    df = pd.read_csv(FEATURES_PATH)
    with open(FEAT_LIST_PATH) as f:
        SELECTED_FEATURES = json.load(f)
    print(f"✅ Loaded engineered dataset from {FEATURES_PATH}")
    print(f"   {df.shape[0]} rows × {df.shape[1]} columns | {len(SELECTED_FEATURES)} selected features")
else:
    print(f"⚠️  {FEATURES_PATH} not found — rebuilding features from {RAW_PATH}")
    df_raw = pd.read_csv(RAW_PATH)
    df = build_features_inline(df_raw)

    SELECTED_FEATURES = [
        'elo_rating', 'squad_value_total_m', 'avg_player_age',
        'players_top5_leagues', 'squad_depth_score',
        'avg_goals_scored_last10', 'avg_goals_conceded_last10',
        'win_rate_last10', 'draw_rate_last10',
        'home_win_rate_last20', 'away_win_rate_last20',
        'clean_sheet_rate', 'avg_opponent_elo_last10', 'xg_difference',
        'unbeaten_streak', 'coach_success_rate',
        'rank_vs_conf', 'elo_vs_conf', 'rank_ratio',
        'form_trajectory', 'form_consistency', 'peak_age_form',
        'goal_ratio', 'defensive_reliability', 'clutch_factor',
        'quality_adjusted_wins', 'team_strength_index',
        'historical_pedigree', 'experience_score', 'coach_efficiency',
        'conf_strength',
    ]
    SELECTED_FEATURES = [f for f in SELECTED_FEATURES if f in df.columns]
    print(f"   Rebuilt {len(SELECTED_FEATURES)} features from raw data")

# ── Verify no missing values ───────────────────────────────────────────────
X = df[SELECTED_FEATURES].copy()
y = df[TARGET].copy()

print(f"\nFeature matrix : {X.shape}")
print(f"Target balance : {y.value_counts().to_dict()}  ({y.mean()*100:.1f}% qualified)")
print(f"Missing values : {X.isnull().sum().sum()}")
df[['country','confederation'] + SELECTED_FEATURES[:5] + [TARGET]].head()

## 3 · Cross-Validation Strategy

With only **100 samples**, hold-out test sets waste precious data.  
We use **Stratified K-Fold (k=5)** as the primary evaluation strategy:

| Parameter | Choice | Rationale |
|-----------|--------|-----------|
| k | 5 | Each fold ~20 samples; small enough for stable estimates |
| Stratified | Yes | Preserves 52/48 class balance in every fold |
| n_repeats | — | Not repeated (added variance for marginal gain at n=100) |
| Test set | 20% hold-out | Kept fully blind until final model evaluation |

**Key metrics:**
- **ROC-AUC** — primary; handles near-balanced classes well
- **F1-score** — harmonic mean of precision and recall
- **Accuracy** — intuitive; valid here given near-equal class sizes

In [ ]:
# ── Stratified split (80 / 20) ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Store team names aligned with test set for later
test_countries = df.loc[X_test.index, 'country'].values
test_confs     = df.loc[X_test.index, 'confederation'].values

print(f"Train : {X_train.shape[0]} samples  | Qualified: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test  : {X_test.shape[0]}  samples  | Qualified: {y_test.sum()}  ({y_test.mean()*100:.1f}%)")

# ── Cross-validation splitter ─────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Scalers (tree models use X_train as-is; linear models need scaling) ───
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std  = scaler_std.transform(X_test)

X_train_std = pd.DataFrame(X_train_std, columns=SELECTED_FEATURES, index=X_train.index)
X_test_std  = pd.DataFrame(X_test_std,  columns=SELECTED_FEATURES, index=X_test.index)

print("\n✅ Train/test split and CV splitter ready.")

In [ ]:
# ── Visualise the split ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# A. Train vs Test class distribution
for ax, data, label in zip(axes[:2], [y_train, y_test], ['Train (80%)', 'Test (20%)']):
    counts = data.value_counts().sort_index()
    bars   = ax.bar(['Not Qual.', 'Qualified'], counts.values,
                    color=PALETTE_QUAL, edgecolor='white', linewidth=1.5, width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', fontweight='bold')
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('Count')
    ax.set_ylim(0, counts.max() * 1.3)

# B. Stratified K-Fold visualisation
fold_sizes = []
for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_sizes.append({'fold': fold_i+1,
                       'train': len(tr_idx), 'val': len(val_idx),
                       'val_pos': y_train.iloc[val_idx].sum()})
fold_df = pd.DataFrame(fold_sizes)

x = np.arange(len(fold_df))
axes[2].bar(x, fold_df['train'], label='Train', color='#90CAF9', edgecolor='white')
axes[2].bar(x, fold_df['val'],   label='Val',   color='#FF8A65', edgecolor='white',
            bottom=fold_df['train'])
axes[2].set_xticks(x)
axes[2].set_xticklabels([f'Fold {i}' for i in fold_df['fold']])
axes[2].set_ylabel('Samples')
axes[2].set_title('Stratified K-Fold (k=5)')
axes[2].legend()

plt.suptitle('Data Split Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Baseline Model

A **Dummy Classifier** (most-frequent-class strategy) gives us the floor.  
Any model that can't beat this is useless.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy_scores = cross_validate(
    dummy, X_train, y_train, cv=cv,
    scoring=['roc_auc', 'f1', 'accuracy'], return_train_score=False
)

print("Dummy Classifier (most-frequent baseline):")
print(f"  ROC-AUC : {dummy_scores['test_roc_auc'].mean():.4f} ± {dummy_scores['test_roc_auc'].std():.4f}")
print(f"  F1      : {dummy_scores['test_f1'].mean():.4f} ± {dummy_scores['test_f1'].std():.4f}")
print(f"  Accuracy: {dummy_scores['test_accuracy'].mean():.4f} ± {dummy_scores['test_accuracy'].std():.4f}")
print("\n→ This is the minimum bar every real model must exceed.")

## 5 · Model Zoo — 6 Classifiers

We evaluate six classifiers covering different inductive biases:

| Model | Scale needed | Non-linear | Interpretable |
|-------|-------------|-----------|---------------|
| Logistic Regression (L2) | ✅ Yes | ❌ No  | ✅ Yes |
| Random Forest | ❌ No | ✅ Yes | 🟡 SHAP |
| XGBoost / GBM | ❌ No | ✅ Yes | 🟡 SHAP |
| Extra Trees | ❌ No | ✅ Yes | 🟡 SHAP |
| SVM (RBF) | ✅ Yes | ✅ Yes | ❌ No |
| KNN | ✅ Yes | ✅ Yes | 🟡 Partial |

In [ ]:
# ── Define model registry ─────────────────────────────────────────────────
boost_model = (
    XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE
    )
    if XGB_AVAILABLE else
    GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, random_state=RANDOM_STATE
    )
)
boost_name = 'XGBoost' if XGB_AVAILABLE else 'GradientBoosting'

MODELS = {
    'LogisticRegression'  : (LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                                                random_state=RANDOM_STATE), True),
    'RandomForest'        : (RandomForestClassifier(n_estimators=300, max_depth=None,
                                                    min_samples_leaf=2, class_weight='balanced',
                                                    random_state=RANDOM_STATE), False),
    boost_name            : (boost_model, False),
    'ExtraTrees'          : (ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2,
                                                  class_weight='balanced',
                                                  random_state=RANDOM_STATE), False),
    'SVM (RBF)'           : (SVC(kernel='rbf', C=1.0, gamma='scale',
                                 probability=True, random_state=RANDOM_STATE), True),
    'KNN'                 : (KNeighborsClassifier(n_neighbors=7, metric='euclidean'), True),
}

# ── Evaluate all models via CV ─────────────────────────────────────────────
scoring_metrics = ['roc_auc', 'f1', 'accuracy', 'precision', 'recall']
cv_results = {}

print("Running 5-Fold Stratified CV for all models...\n")
for name, (model, needs_scale) in MODELS.items():
    X_cv = X_train_std if needs_scale else X_train
    scores = cross_validate(model, X_cv, y_train, cv=cv,
                            scoring=scoring_metrics, return_train_score=True)
    cv_results[name] = scores
    auc   = scores['test_roc_auc'].mean()
    f1    = scores['test_f1'].mean()
    acc   = scores['test_accuracy'].mean()
    print(f"  {name:<22} AUC={auc:.4f} ± {scores['test_roc_auc'].std():.4f}  "
          f"F1={f1:.4f}  Acc={acc:.4f}")

print("\n✅ CV complete.")

In [ ]:
# ── CV Results Summary DataFrame ─────────────────────────────────────────
summary_rows = []
for name, scores in cv_results.items():
    summary_rows.append({
        'Model'          : name,
        'ROC-AUC (mean)' : scores['test_roc_auc'].mean(),
        'ROC-AUC (std)'  : scores['test_roc_auc'].std(),
        'F1 (mean)'      : scores['test_f1'].mean(),
        'Accuracy (mean)': scores['test_accuracy'].mean(),
        'Precision (mean)': scores['test_precision'].mean(),
        'Recall (mean)'  : scores['test_recall'].mean(),
        'Train AUC'      : scores['train_roc_auc'].mean(),
        'Overfit gap'    : scores['train_roc_auc'].mean() - scores['test_roc_auc'].mean(),
    })

cv_summary = pd.DataFrame(summary_rows).sort_values('ROC-AUC (mean)', ascending=False).reset_index(drop=True)
cv_summary.style \
    .format({'ROC-AUC (mean)': '{:.4f}', 'ROC-AUC (std)': '{:.4f}',
             'F1 (mean)': '{:.4f}', 'Accuracy (mean)': '{:.4f}',
             'Precision (mean)': '{:.4f}', 'Recall (mean)': '{:.4f}',
             'Train AUC': '{:.4f}', 'Overfit gap': '{:+.4f}'}) \
    .background_gradient(subset=['ROC-AUC (mean)', 'F1 (mean)'], cmap='YlOrRd') \
    .background_gradient(subset=['Overfit gap'], cmap='RdYlGn_r')

In [ ]:
# ── Visualise CV comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['ROC-AUC (mean)', 'F1 (mean)', 'Accuracy (mean)']
titles  = ['ROC-AUC', 'F1-Score', 'Accuracy']
colors  = sns.color_palette('viridis', len(cv_summary))

for ax, metric, title in zip(axes, metrics, titles):
    sorted_df = cv_summary.sort_values(metric)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric],
                   color=colors, edgecolor='white', linewidth=0.8)
    # Add error bars for AUC
    if metric == 'ROC-AUC (mean)':
        ax.errorbar(sorted_df[metric], range(len(sorted_df)),
                    xerr=sorted_df['ROC-AUC (std)'], fmt='none',
                    color='black', capsize=4, linewidth=1.2)
    ax.axvline(dummy_scores[f'test_{title.lower().replace("-","_").replace(" ","_")}'
                             .replace('roc_auc', 'roc_auc').replace('f1_score','f1')].mean()
               if metric != 'Accuracy (mean)' else dummy_scores['test_accuracy'].mean(),
               color='red', linestyle='--', linewidth=1.2, label='Dummy baseline',
               alpha=0.7)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, 1.08)
    if ax == axes[0]: ax.legend(fontsize=8)

plt.suptitle('Model Comparison — 5-Fold Stratified Cross-Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Overfitting analysis ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x     = np.arange(len(cv_summary))
width = 0.35

ax.bar(x - width/2, cv_summary['Train AUC'],        width, label='Train AUC',      color='#1976D2', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, cv_summary['ROC-AUC (mean)'],   width, label='Val AUC (CV)',   color='#FF5722', alpha=0.85, edgecolor='white')

for xi, (train, val) in enumerate(zip(cv_summary['Train AUC'], cv_summary['ROC-AUC (mean)'])):
    gap = train - val
    color = '#d32f2f' if gap > 0.15 else '#f9a825' if gap > 0.05 else '#388e3c'
    ax.annotate('', xy=(xi + width/2, val), xytext=(xi - width/2, train),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

ax.set_xticks(x)
ax.set_xticklabels(cv_summary['Model'], rotation=15, ha='right')
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.4, 1.05)
ax.set_title('Train vs Validation AUC — Overfitting Audit', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

best_model_name = cv_summary.iloc[0]['Model']
print(f"🏆 Best model by CV ROC-AUC: {best_model_name}  ({cv_summary.iloc[0]['ROC-AUC (mean)']:.4f})")

## 6 · Hyperparameter Tuning

We tune the **top two models** (typically Random Forest and XGBoost/GBM) using **RandomizedSearchCV** (more efficient than GridSearch for large spaces).

All tuning is done on **training data only** — test set remains blind.

In [ ]:
# ── Tune Random Forest ────────────────────────────────────────────────────
rf_param_dist = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [None, 3, 5, 8, 12],
    'min_samples_leaf': [1, 2, 3, 5],
    'min_samples_split': [2, 5, 10],
    'max_features'    : ['sqrt', 'log2', 0.5, 0.7],
    'class_weight'    : ['balanced', None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_dist,
    n_iter=60,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)
rf_search.fit(X_train, y_train)

print("Random Forest — Best hyperparameters:")
for k, v in rf_search.best_params_.items():
    print(f"  {k:<25} : {v}")
print(f"\n  Best CV AUC : {rf_search.best_score_:.4f}")

In [ ]:
# ── Tune XGBoost / GBM ───────────────────────────────────────────────────
if XGB_AVAILABLE:
    boost_param_dist = {
        'n_estimators'     : [100, 200, 300, 500],
        'learning_rate'    : [0.01, 0.03, 0.05, 0.1, 0.15],
        'max_depth'        : [2, 3, 4, 5, 6],
        'subsample'        : [0.6, 0.7, 0.8, 1.0],
        'colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
        'gamma'            : [0, 0.1, 0.5, 1.0],
        'reg_alpha'        : [0, 0.1, 0.5, 1.0],
        'reg_lambda'       : [1, 2, 5],
        'min_child_weight' : [1, 3, 5],
    }
    boost_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                               random_state=RANDOM_STATE)
else:
    boost_param_dist = {
        'n_estimators'  : [100, 200, 300],
        'learning_rate' : [0.01, 0.05, 0.1, 0.15],
        'max_depth'     : [2, 3, 4, 5],
        'subsample'     : [0.7, 0.8, 1.0],
        'min_samples_leaf': [1, 2, 3],
    }
    boost_base = GradientBoostingClassifier(random_state=RANDOM_STATE)

boost_search = RandomizedSearchCV(
    boost_base, boost_param_dist,
    n_iter=60, cv=cv, scoring='roc_auc',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
boost_search.fit(X_train, y_train)

print(f"{boost_name} — Best hyperparameters:")
for k, v in boost_search.best_params_.items():
    print(f"  {k:<25} : {v}")
print(f"\n  Best CV AUC : {boost_search.best_score_:.4f}")

In [ ]:
# ── Tuning improvement summary ────────────────────────────────────────────
rf_base_auc   = cv_results['RandomForest']['test_roc_auc'].mean()
boost_base_auc = cv_results[boost_name]['test_roc_auc'].mean()

print("Hyperparameter Tuning Lift:")
print(f"  Random Forest  : {rf_base_auc:.4f} → {rf_search.best_score_:.4f}  "
      f"({'+'if rf_search.best_score_ > rf_base_auc else ''}{rf_search.best_score_ - rf_base_auc:+.4f})")
print(f"  {boost_name:<15}: {boost_base_auc:.4f} → {boost_search.best_score_:.4f}  "
      f"({boost_search.best_score_ - boost_base_auc:+.4f})")

## 7 · Model Selection & Ensemble

We also build a **soft-voting ensemble** of the top 3 models.  
On small datasets, ensembles often beat any single model due to variance reduction.

In [ ]:
# ── Refit top models with tuned params ───────────────────────────────────
best_rf    = rf_search.best_estimator_
best_boost = boost_search.best_estimator_
best_lr    = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)

# ── Soft-voting ensemble ──────────────────────────────────────────────────
ensemble = VotingClassifier(
    estimators=[
        ('rf',    best_rf),
        ('boost', best_boost),
        ('lr',    Pipeline([('scaler', StandardScaler()), ('lr', best_lr)])),
    ],
    voting='soft',
    weights=[2, 2, 1]    # RF and Boost get more weight
)

ensemble_cv = cross_val_score(ensemble, X_train, y_train, cv=cv,
                              scoring='roc_auc', n_jobs=-1)

print("Ensemble (soft voting) CV AUC:")
print(f"  Mean: {ensemble_cv.mean():.4f} ± {ensemble_cv.std():.4f}")
print(f"  Per fold: {np.round(ensemble_cv, 4)}")

all_tuned_results = {
    'RandomForest (tuned)' : rf_search.best_score_,
    f'{boost_name} (tuned)': boost_search.best_score_,
    'Ensemble (soft vote)' : ensemble_cv.mean(),
}
for name, score in sorted(all_tuned_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<28} AUC = {score:.4f}")

# ── Select final model ────────────────────────────────────────────────────
FINAL_MODEL_NAME = max(all_tuned_results, key=all_tuned_results.get)
if FINAL_MODEL_NAME == 'Ensemble (soft vote)':
    FINAL_MODEL = ensemble
elif 'RandomForest' in FINAL_MODEL_NAME:
    FINAL_MODEL = best_rf
else:
    FINAL_MODEL = best_boost

print(f"\n🏆 Selected final model: {FINAL_MODEL_NAME}")

## 8 · Final Model Evaluation on Hold-Out Test Set

The test set has been **completely untouched** until now.  
These are our unbiased performance estimates.

In [ ]:
# ── Fit final model on ALL training data, then evaluate on test ───────────
FINAL_MODEL.fit(X_train, y_train)

y_pred       = FINAL_MODEL.predict(X_test)
y_prob       = FINAL_MODEL.predict_proba(X_test)[:, 1]

test_auc  = roc_auc_score(y_test, y_prob)
test_f1   = f1_score(y_test, y_pred)
test_acc  = accuracy_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred)
test_rec  = recall_score(y_test, y_pred)

print("═" * 50)
print(f"  FINAL MODEL: {FINAL_MODEL_NAME}")
print("═" * 50)
print(f"  ROC-AUC   : {test_auc:.4f}")
print(f"  F1-Score  : {test_f1:.4f}")
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  Precision : {test_prec:.4f}")
print(f"  Recall    : {test_rec:.4f}")
print("═" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Qualified', 'Qualified']))

In [ ]:
# ── Comprehensive evaluation plots ────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])   # Confusion Matrix
ax2 = fig.add_subplot(gs[0, 1])   # ROC Curve
ax3 = fig.add_subplot(gs[0, 2])   # Precision-Recall Curve
ax4 = fig.add_subplot(gs[1, 0])   # Probability distribution
ax5 = fig.add_subplot(gs[1, 1])   # Calibration curve
ax6 = fig.add_subplot(gs[1, 2])   # Metric summary bar

# A. Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Not Qual.', 'Qualified'],
            yticklabels=['Not Qual.', 'Qualified'],
            linewidths=1, linecolor='white')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix')

# B. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#FF5722', linewidth=2, label=f'AUC = {test_auc:.4f}')
ax2.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax2.fill_between(fpr, tpr, alpha=0.1, color='#FF5722')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve'); ax2.legend()

# C. Precision-Recall
from sklearn.metrics import precision_recall_curve, average_precision_score
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax3.plot(rec_curve, prec_curve, color='#2196F3', linewidth=2, label=f'AP = {ap:.4f}')
ax3.axhline(y_test.mean(), color='red', linestyle='--', linewidth=1, alpha=0.7, label='Baseline')
ax3.fill_between(rec_curve, prec_curve, alpha=0.1, color='#2196F3')
ax3.set_xlabel('Recall'); ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall Curve'); ax3.legend()

# D. Predicted probability distributions
ax4.hist(y_prob[y_test == 0], bins=15, color=PALETTE_QUAL[0], alpha=0.7,
         label='Not Qualified', density=True, edgecolor='white')
ax4.hist(y_prob[y_test == 1], bins=15, color=PALETTE_QUAL[1], alpha=0.7,
         label='Qualified',    density=True, edgecolor='white')
ax4.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold 0.5')
ax4.set_xlabel('Predicted Probability of Qualifying')
ax4.set_title('Predicted Probability Distributions'); ax4.legend()

# E. Calibration curve
try:
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=5, strategy='uniform')
    ax5.plot(prob_pred, prob_true, 'o-', color='#FF5722', linewidth=2, label=FINAL_MODEL_NAME)
    ax5.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Perfect calibration')
    ax5.set_xlabel('Mean Predicted Probability'); ax5.set_ylabel('Fraction of Positives')
    ax5.set_title('Calibration Curve'); ax5.legend(fontsize=8)
except Exception as e:
    ax5.text(0.5, 0.5, f'Calibration\nnot available\n({str(e)[:30]})',
             ha='center', va='center', transform=ax5.transAxes)

# F. Metrics bar
metric_names = ['ROC-AUC', 'F1', 'Accuracy', 'Precision', 'Recall']
metric_vals  = [test_auc, test_f1, test_acc, test_prec, test_rec]
bars = ax6.bar(metric_names, metric_vals,
               color=['#FF5722','#1976D2','#388E3C','#7B1FA2','#F57C00'],
               edgecolor='white', linewidth=1.2, width=0.55)
for bar, val in zip(bars, metric_vals):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
ax6.set_ylim(0, 1.15); ax6.set_ylabel('Score')
ax6.set_title('Test-Set Metrics Summary')

plt.suptitle(f'Final Model Evaluation — {FINAL_MODEL_NAME}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Error analysis — what did the model get wrong? ────────────────────────
test_results = pd.DataFrame({
    'country'       : test_countries,
    'confederation' : test_confs,
    'actual'        : y_test.values,
    'predicted'     : y_pred,
    'prob_qualify'  : y_prob,
})
test_results['correct']       = (test_results['actual'] == test_results['predicted']).astype(int)
test_results['error_type']    = 'Correct'
test_results.loc[(test_results['actual']==0) & (test_results['predicted']==1), 'error_type'] = 'FP (predicted qualify)'
test_results.loc[(test_results['actual']==1) & (test_results['predicted']==0), 'error_type'] = 'FN (missed qualify)'

mistakes = test_results[test_results['correct'] == 0].sort_values('prob_qualify', ascending=False)
print(f"Misclassified teams ({len(mistakes)} / {len(test_results)}):")
display(mistakes[['country','confederation','actual','predicted','prob_qualify','error_type']]
        .style.format({'prob_qualify': '{:.3f}'})
        .background_gradient(subset=['prob_qualify'], cmap='RdYlGn'))

print(f"\n  False Positives (predicted qualified, didn't): {(mistakes['error_type'] == 'FP (predicted qualify)').sum()}")
print(f"  False Negatives (missed qualified teams)     : {(mistakes['error_type'] == 'FN (missed qualify)').sum()}")

## 9 · SHAP Explainability

SHAP (SHapley Additive exPlanations) provides **globally consistent, locally accurate** feature attributions.  
For tree-based models, SHAP uses the fast TreeSHAP algorithm (exact, O(TLD²) complexity).

In [ ]:
if SHAP_AVAILABLE:
    # Use the tree-based model (RF or Boost) for SHAP — skip if ensemble selected
    shap_model = best_boost if hasattr(best_boost, 'feature_importances_') else best_rf
    shap_model.fit(X_train, y_train)

    explainer   = shap.TreeExplainer(shap_model)
    shap_values = explainer.shap_values(X)

    # For multi-output (RF returns array of arrays), take the positive class
    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    print(f"SHAP values computed for {sv.shape[0]} samples × {sv.shape[1]} features")
else:
    print("⚠️  SHAP not available — skipping Section 9.")
    print("    Install with: pip install shap")

In [ ]:
if SHAP_AVAILABLE:
    # ── Global SHAP summary plots ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 9))

    # A. Bar plot — mean absolute SHAP
    mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=SELECTED_FEATURES).sort_values(ascending=True)
    top20     = mean_shap.tail(20)
    colors_shap = ['#FF5722' if v in mean_shap.tail(5).index else '#90A4AE' for v in top20.index]
    axes[0].barh(top20.index, top20.values, color=colors_shap, edgecolor='white')
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].set_title('Top 20 Features — Global SHAP Importance', fontweight='bold')

    # B. SHAP beeswarm (dot plot)
    plt.sca(axes[1])
    shap.summary_plot(sv, X, feature_names=SELECTED_FEATURES,
                      max_display=20, show=False, plot_type='dot')
    axes[1].set_title('SHAP Beeswarm — Feature Direction & Magnitude', fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
if SHAP_AVAILABLE:
    # ── SHAP dependence plots for top 4 features ──────────────────────────
    top4_features = mean_shap.tail(4).index.tolist()[::-1]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, feat in zip(axes, top4_features):
        feat_idx = list(X.columns).index(feat) if hasattr(X, 'columns') else SELECTED_FEATURES.index(feat)
        ax.scatter(
            X[feat] if hasattr(X, '__getitem__') else X[:, feat_idx],
            sv[:, feat_idx],
            c=y.values, cmap='RdYlGn', alpha=0.7, s=40, edgecolors='white', linewidth=0.3
        )
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel(feat.replace('_', ' ').title(), fontsize=9)
        ax.set_ylabel('SHAP value', fontsize=9)
        ax.set_title(feat.replace('_', ' ').title(), fontsize=9)

    plt.suptitle('SHAP Dependence Plots — Top 4 Features\n(colour = Qualified: green / Not Qualified: red)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if SHAP_AVAILABLE:
    # ── SHAP waterfall — top 3 correctly predicted qualifiers ─────────────
    # test_results was built with X_test.index, so use that directly
    correct_q = test_results[(test_results['actual']==1) & (test_results['correct']==1)]
    showcase  = correct_q.sort_values('prob_qualify', ascending=False).head(3)

    base_val = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, (list, np.ndarray))
        else explainer.expected_value
    )
    X_index_list = list(X.index)   # full dataset index for SHAP lookup

    print("SHAP waterfall explanations — top 3 correctly predicted qualified teams:")
    for orig_df_idx, row in showcase.iterrows():
        # orig_df_idx is the original DataFrame index (e.g. 18, 42…)
        if orig_df_idx not in X_index_list:
            continue
        full_idx = X_index_list.index(orig_df_idx)
        country  = df.loc[orig_df_idx, 'country'] if orig_df_idx in df.index else row['country']
        print(f"\n  {country} (p={row['prob_qualify']:.3f})")
        exp = shap.Explanation(
            values        = sv[full_idx],
            base_values   = base_val,
            data          = X.iloc[full_idx].values,
            feature_names = SELECTED_FEATURES
        )
        shap.plots.waterfall(exp, max_display=12, show=True)

## 10 · Confederation-Level Analysis

Breaking down predictions by confederation reveals how well the model handles the structural differences between FIFA's regional bodies.

In [ ]:
# ── Run predictions on full dataset for confederation analysis ────────────
FINAL_MODEL.fit(X_train, y_train)   # ensure fitted
df['prob_qualify'] = FINAL_MODEL.predict_proba(X)[:, 1]
df['predicted']    = FINAL_MODEL.predict(X)

conf_metrics = []
for conf in CONF_ORDER:
    mask      = df['confederation'] == conf
    y_c       = df.loc[mask, TARGET]
    pred_c    = df.loc[mask, 'predicted']
    prob_c    = df.loc[mask, 'prob_qualify']
    n_total   = mask.sum()
    n_qual    = y_c.sum()
    n_pred    = pred_c.sum()

    try:
        conf_auc = roc_auc_score(y_c, prob_c) if len(y_c.unique()) > 1 else np.nan
        conf_acc = accuracy_score(y_c, pred_c)
    except Exception:
        conf_auc = conf_acc = np.nan

    conf_metrics.append({
        'Confederation'   : conf,
        'N teams'         : n_total,
        'Actual qualified': n_qual,
        'Pred qualified'  : n_pred,
        'Qual rate'       : f"{n_qual/n_total*100:.0f}%",
        'ROC-AUC'         : conf_auc,
        'Accuracy'        : conf_acc,
    })

conf_df = pd.DataFrame(conf_metrics)
display(conf_df.style.format({
    'ROC-AUC': '{:.3f}', 'Accuracy': '{:.3f}'
}).background_gradient(subset=['ROC-AUC','Accuracy'], cmap='YlOrRd'))

In [ ]:
# ── Confederation probability distributions ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, conf, color in zip(axes.flatten(), CONF_ORDER, PALETTE_CONF):
    mask  = df['confederation'] == conf
    df_c  = df[mask].sort_values('prob_qualify', ascending=False)

    bar_colors = [PALETTE_QUAL[1] if q == 1 else PALETTE_QUAL[0] for q in df_c[TARGET]]
    ax.bar(range(len(df_c)), df_c['prob_qualify'],
           color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.6)

    ax.set_title(conf, fontweight='bold', color=color)
    ax.set_xlabel('Teams (sorted by P(qualify))')
    ax.set_ylabel('P(qualify)')
    ax.set_ylim(0, 1.1)

    # Annotate team names for high-confidence predictions
    for i, (_, row) in enumerate(df_c.iterrows()):
        if row['prob_qualify'] > 0.8 or row['prob_qualify'] < 0.2:
            ax.text(i, row['prob_qualify'] + 0.02, row['country'][:3],
                    ha='center', fontsize=6, rotation=45)

legend_els = [mpatches.Patch(color=PALETTE_QUAL[1], label='Actual: Qualified'),
              mpatches.Patch(color=PALETTE_QUAL[0], label='Actual: Not Qualified')]
fig.legend(handles=legend_els, loc='upper right', bbox_to_anchor=(1.02, 1.0))

plt.suptitle('Predicted Qualification Probability by Confederation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11 · World Cup 2026 Predictions

Full ranking of all 100 teams by predicted probability of qualification,  
with confederations and confidence bands.

In [ ]:
# ── Build prediction table ────────────────────────────────────────────────
predictions = df[['country', 'confederation', TARGET, 'prob_qualify', 'predicted']].copy()
predictions = predictions.sort_values('prob_qualify', ascending=False).reset_index(drop=True)
predictions.index = predictions.index + 1
predictions.index.name = 'Rank'

predictions['Confidence'] = pd.cut(
    predictions['prob_qualify'],
    bins=[0, 0.30, 0.45, 0.55, 0.70, 1.0],
    labels=['Strong NO', 'Lean NO', 'Uncertain', 'Lean YES', 'Strong YES']
)
predictions['Match'] = (predictions[TARGET] == predictions['predicted']).map({True: '✅', False: '❌'})

print("Top 20 — Highest probability of qualifying for WC2026:")
display(
    predictions.head(20)
    [['country','confederation','prob_qualify','Confidence',TARGET,'predicted','Match']]
    .rename(columns={
        'country': 'Country', 'confederation': 'Conf',
        'prob_qualify': 'P(qualify)', TARGET: 'Actual', 'predicted': 'Predicted'
    })
    .style.format({'P(qualify)': '{:.3f}'})
    .background_gradient(subset=['P(qualify)'], cmap='YlOrRd')
)

In [ ]:
# ── Full team ranking plot ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 28))

bar_colors = [PALETTE_QUAL[1] if q == 1 else PALETTE_QUAL[0]
              for q in predictions[TARGET]]
bars = ax.barh(
    range(len(predictions)),
    predictions['prob_qualify'],
    color=bar_colors, edgecolor='white', linewidth=0.5, height=0.8
)

ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.7)

ax.set_yticks(range(len(predictions)))
ax.set_yticklabels(
    [f"{row.country} ({row.confederation})" for _, row in predictions.iterrows()],
    fontsize=7.5
)
ax.invert_yaxis()
ax.set_xlabel('Predicted Probability of Qualifying for WC2026')
ax.set_title('WC2026 Qualification Probability — All 100 Teams',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.1)

# Annotate probability values
for i, prob in enumerate(predictions['prob_qualify']):
    ax.text(prob + 0.01, i, f'{prob:.2f}', va='center', fontsize=6.5)

legend_els = [mpatches.Patch(color=PALETTE_QUAL[1], label='Actual: Qualified'),
              mpatches.Patch(color=PALETTE_QUAL[0], label='Actual: Not Qualified')]
ax.legend(handles=legend_els, loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Confidence breakdown ──────────────────────────────────────────────────
conf_band = predictions['Confidence'].value_counts().reindex(
    ['Strong YES', 'Lean YES', 'Uncertain', 'Lean NO', 'Strong NO']
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# A. Band counts
palette = ['#1B5E20', '#66BB6A', '#FFA726', '#EF5350', '#B71C1C']
bars = axes[0].bar(conf_band.index, conf_band.values, color=palette, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, conf_band.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(v), ha='center', fontweight='bold')
axes[0].set_ylabel('Number of Teams')
axes[0].set_title('Teams per Confidence Band')
axes[0].tick_params(axis='x', rotation=15)

# B. Accuracy within each band
band_acc = (
    predictions.groupby('Confidence')
    .apply(lambda g: (g[TARGET] == g['predicted']).mean())
    .reindex(['Strong YES', 'Lean YES', 'Uncertain', 'Lean NO', 'Strong NO'])
)
bars2 = axes[1].bar(band_acc.index, band_acc.values, color=palette, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars2, band_acc.values):
    if not np.isnan(v):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{v:.0%}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel('Accuracy within band')
axes[1].set_title('Accuracy per Confidence Band')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Prediction Confidence Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12 · Results Export

In [ ]:
# ── Save model results ────────────────────────────────────────────────────
# 1. CV model comparison table
cv_summary.to_csv('wc2026_model_results.csv', index=False)
print("✅ Saved: wc2026_model_results.csv")

# 2. Full team predictions
predictions_out = predictions.reset_index()[[
    'Rank', 'country', 'confederation', 'prob_qualify',
    'Confidence', TARGET, 'predicted', 'Match'
]].rename(columns={'country': 'Country', 'confederation': 'Confederation',
                   'prob_qualify': 'P_qualify', TARGET: 'Actual_qualified',
                   'predicted': 'Predicted_qualified'})
predictions_out.to_csv('wc2026_final_predictions.csv', index=False)
print("✅ Saved: wc2026_final_predictions.csv")

# 3. Save feature importances (from best tree model)
try:
    tree_model = best_boost if hasattr(best_boost, 'feature_importances_') else best_rf
    tree_model.fit(X_train, y_train)
    feat_imp = pd.DataFrame({
        'feature'   : SELECTED_FEATURES,
        'importance': tree_model.feature_importances_
    }).sort_values('importance', ascending=False)
    feat_imp.to_csv('wc2026_feature_importances.csv', index=False)
    print("✅ Saved: wc2026_feature_importances.csv")
except Exception as e:
    print(f"⚠️  Feature importance export skipped: {e}")

# 4. Summary stats
print("\n" + "="*55)
print("  PIPELINE SUMMARY")
print("="*55)
print(f"  Dataset         : 100 teams, {len(SELECTED_FEATURES)} selected features")
print(f"  Best model (CV) : {FINAL_MODEL_NAME}")
print(f"  Test ROC-AUC    : {test_auc:.4f}")
print(f"  Test F1-Score   : {test_f1:.4f}")
print(f"  Test Accuracy   : {test_acc:.4f}")
correct_total = (predictions[TARGET] == predictions['predicted']).sum()
print(f"  Full-data accuracy: {correct_total}/{len(predictions)} ({correct_total/len(predictions)*100:.1f}%)")
print("="*55)

---
## ✅ Modelling Complete

**Output files:**
- `wc2026_model_results.csv` — CV comparison across all 6 models
- `wc2026_final_predictions.csv` — qualification probability for all 100 teams
- `wc2026_feature_importances.csv` — feature importance from best tree model

**Pipeline recap:**
1. EDA (`01_eda.ipynb`) — distributions, correlations, feature hypotheses
2. Feature Engineering (`02_feature_engineering.ipynb`) — 15 engineered features, selection
3. Modelling (`03_modelling.ipynb`) — 6 models, tuning, SHAP, confederation analysis

*FIFA World Cup 2026 Analytics & Prediction Pipeline — End of notebook.*